# Datetime and Time Series

Dates and times are everywhere: orders, logs, sensors, prices, users signing up. They are also where the **most expensive ML mistakes** happen (using the future to predict the past, wrong time zones, random splits on time data).

**Contents**
1. Datetime types in pandas
2. Creating and parsing dates
3. The `.dt` accessor (extract parts)
4. Date arithmetic: `Timedelta`, offsets, business days
5. Selecting by date: `DatetimeIndex`, slicing, time of day
6. `resample`: change the frequency
7. Missing dates and gaps
8. Rolling windows by time, `shift`, `diff`
9. Time zones
10. Date features for ML (calendar, cyclical, holidays)
11. Mini AI/ML example: demand forecasting with a **time-based split**
12. Pandas 3.x notes
13. Common mistakes
14. Interview questions
15. Practice questions with solutions
16. Cheat sheet

In [1]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)

pandas: 3.0.6


---
# 1. Datetime types in pandas

| Type | Meaning | Example |
|---|---|---|
| `Timestamp` | one point in time | `2026-03-15 10:30:00` |
| `DatetimeIndex` / `datetime64` column | many timestamps | a column of order times |
| `Timedelta` | a **duration** | `3 days 04:00:00` |
| `Period` | a **span** (a month, a quarter) | `2026-03` (the whole March) |
| `DateOffset` | a calendar step | "1 month", "3 business days" |
| `NaT` | missing time ("Not a Time") | like NaN for dates |

The most important rule: a date column should use a pandas datetime-like dtype
(such as datetime64 or a timezone-aware datetime dtype). If it is text (`str`),
date arithmetic and time-aware operations will not work as intended.

---
# 2. Creating and parsing dates

In [2]:
ts = pd.Timestamp("2026-03-15 10:30:00")
print(ts)
print(ts.year, ts.month, ts.day, ts.hour, ts.day_name())

2026-03-15 10:30:00
2026 3 15 10 Sunday


In [3]:
# Text -> datetime (chapter 08 covered messy formats)
s = pd.Series(["2026-03-01", "2026-03-15", "2026-04-02"])
print(s.dtype)

d = pd.to_datetime(s)
print(d.dtype)
d

str
datetime64[us]


0   2026-03-01
1   2026-03-15
2   2026-04-02
dtype: datetime64[us]

In [4]:
# Explicit format is the safest and fastest when you know it
print(pd.to_datetime("15/03/2026", format="%d/%m/%Y"))
print(pd.to_datetime("March 15, 2026", format="%B %d, %Y"))

# Bad values: errors="coerce" gives NaT
print(pd.to_datetime(["2026-03-01", "not a date"], errors="coerce").tolist())

2026-03-15 00:00:00
2026-03-15 00:00:00
[Timestamp('2026-03-01 00:00:00'), NaT]


Common format codes: `%Y` year (2026), `%y` (26), `%m` month number, `%B` month name, `%b` (Mar), `%d` day, `%H` hour (24h), `%I` hour (12h), `%M` minute, `%S` second, `%p` AM/PM, `%A` weekday name.

In [5]:
# Unix timestamps (seconds since 1970-01-01) are common in APIs and logs
print(pd.to_datetime([1772323200, 1772409600], unit="s").tolist())

[Timestamp('2026-03-01 00:00:00'), Timestamp('2026-03-02 00:00:00')]


In [6]:
# Separate columns -> one datetime
parts = pd.DataFrame({"year": [2026, 2026], "month": [3, 4], "day": [15, 2]})
pd.to_datetime(parts)

0   2026-03-15
1   2026-04-02
dtype: datetime64[us]

In [7]:
# Reading a CSV: parse_dates converts while loading (chapter 03)
from io import StringIO

csv = StringIO("date,sales\n2026-03-01,100\n2026-03-02,120\n")
df = pd.read_csv(csv, parse_dates=["date"])
print(df.dtypes)

date     datetime64[us]
sales             int64
dtype: object


## `date_range`: make a regular sequence of dates

In [8]:
pd.date_range("2026-03-01", periods=5)                       # 5 daily dates

DatetimeIndex(['2026-03-01', '2026-03-02', '2026-03-03', '2026-03-04',
               '2026-03-05'],
              dtype='datetime64[us]', freq='D')

In [9]:
print(pd.date_range("2026-03-01", "2026-03-31", freq="W").tolist()[:3])        # weekly (Sundays)
print(pd.date_range("2026-01-31", periods=4, freq="ME").tolist())              # month ends
print(pd.date_range("2026-03-01 09:00", periods=4, freq="h").tolist())         # hourly
print(pd.date_range("2026-03-01", periods=5, freq="B").day_name().tolist())    # business days (Mon-Fri)

[Timestamp('2026-03-01 00:00:00'), Timestamp('2026-03-08 00:00:00'), Timestamp('2026-03-15 00:00:00')]
[Timestamp('2026-01-31 00:00:00'), Timestamp('2026-02-28 00:00:00'), Timestamp('2026-03-31 00:00:00'), Timestamp('2026-04-30 00:00:00')]
[Timestamp('2026-03-01 09:00:00'), Timestamp('2026-03-01 10:00:00'), Timestamp('2026-03-01 11:00:00'), Timestamp('2026-03-01 12:00:00')]
['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']


## Frequency aliases (pandas 3)

| Alias | Meaning | | Alias | Meaning |
|---|---|---|---|---|
| `"D"` | calendar day | | `"h"` | hour |
| `"B"` | business day | | `"min"` | minute |
| `"W"` | week (ends Sunday) | | `"s"` | second |
| `"ME"` | **month end** | | `"MS"` | month **start** |
| `"QE"` | **quarter end** | | `"YE"` | **year end** |

The old `"M"`, `"Q"`, `"Y"`, `"H"`, `"T"` are **no longer accepted** in pandas 3. Use `"ME"`, `"QE"`, `"YE"`, `"h"`, `"min"`.

---
# 3. The `.dt` accessor

Once a column has a pandas datetime-like dtype, `.dt` gives its date/time parts.

In [10]:
orders = pd.DataFrame({
    "order_time": ["2026-03-01 09:15:00", "2026-03-02 14:40:30", "2026-03-05 23:59:59", "2026-03-06 00:05:10",
                   "2026-03-08 18:20:00", "2026-03-13 11:00:00", "2026-03-14 08:30:45", "2026-03-15 20:10:00"],
    "amount": [250, 120, 980, 60, 430, 310, 75, 540],
})
orders["order_time"] = pd.to_datetime(orders["order_time"])
orders.dtypes

order_time    datetime64[us]
amount                 int64
dtype: object

In [11]:
t = orders["order_time"].dt

parts = pd.DataFrame({
    "time":       orders["order_time"],
    "year":       t.year,
    "month":      t.month,
    "month_name": t.month_name(),
    "day":        t.day,
    "weekday_no": t.dayofweek,          # Monday = 0 ... Sunday = 6
    "weekday":    t.day_name(),
    "hour":       t.hour,
    "quarter":    t.quarter,
    "day_of_year": t.dayofyear,
    "is_weekend": t.dayofweek >= 5,
})
parts

,time,year,month,month_name,day,weekday_no,weekday,hour,quarter,day_of_year,is_weekend
0,2026-03-01 09:15:00,2026,3,March,1,6,Sunday,9,1,60,True
1,2026-03-02 14:40:30,2026,3,March,2,0,Monday,14,1,61,False
2,2026-03-05 23:59:59,2026,3,March,5,3,Thursday,23,1,64,False
3,2026-03-06 00:05:10,2026,3,March,6,4,Friday,0,1,65,False
4,2026-03-08 18:20:00,2026,3,March,8,6,Sunday,18,1,67,True
5,2026-03-13 11:00:00,2026,3,March,13,4,Friday,11,1,72,False
6,2026-03-14 08:30:45,2026,3,March,14,5,Saturday,8,1,73,True
7,2026-03-15 20:10:00,2026,3,March,15,6,Sunday,20,1,74,True


In [12]:
t = orders["order_time"].dt

extra = pd.DataFrame({
    "is_month_end": t.is_month_end,
    "week_of_year": t.isocalendar().week,        # (dt.week was removed)
    "date_only":    t.date,                      # Python date objects (object dtype, avoid for maths)
    "midnight":     t.normalize(),               # same day at 00:00, stays datetime64 (better than .date)
    "hour_floor":   t.floor("h"),                # round DOWN to the hour
    "text":         t.strftime("%d %b %Y, %I:%M %p"),    # format as text for display
})
extra.head(4)

,is_month_end,week_of_year,date_only,midnight,hour_floor,text
0,False,9,2026-03-01,2026-03-01,2026-03-01 09:00:00,"01 Mar 2026, 09:15 AM"
1,False,10,2026-03-02,2026-03-02,2026-03-02 14:00:00,"02 Mar 2026, 02:40 PM"
2,False,10,2026-03-05,2026-03-05,2026-03-05 23:00:00,"05 Mar 2026, 11:59 PM"
3,False,10,2026-03-06,2026-03-06,2026-03-06 00:00:00,"06 Mar 2026, 12:05 AM"


- Use **`normalize()`** (or `floor("D")`) instead of `.dt.date` when you need to group by day, because it keeps the `datetime64` dtype.
- `strftime` returns **text**. Use it only for display or file names, not for further date maths.
- Weekday numbering: **Monday = 0 ... Sunday = 6**.

---
# 4. Date arithmetic

## 4.1 Subtracting dates gives a `Timedelta`

In [13]:
a = pd.Timestamp("2026-03-15 10:00")
b = pd.Timestamp("2026-01-01 08:30")

diff = a - b
print(diff)
print("days         :", diff.days)
print("total hours  :", round(diff.total_seconds() / 3600, 1))

73 days 01:30:00
days         : 73
total hours  : 1753.5


In [66]:
# On a column: tenure in days
people = pd.DataFrame({
    "name": ["Ravi", "Asha", "Karan"],
    "joined": pd.to_datetime(["2024-06-10", "2025-01-25", "2026-02-14"]),
})
today = pd.Timestamp("2026-03-15")                # fixed date so the notebook always gives the same output

people["tenure_days"] = (today - people["joined"]).dt.days

# approximate months using average month length
people["tenure_months"] = (people["tenure_days"] / 30.44).round(1)

people["tenure_years"] = (people["tenure_days"] / 365.25).round(2)
people

,name,joined,tenure_days,tenure_months,tenure_years
0,Ravi,2024-06-10,643,21.1,1.76
1,Asha,2025-01-25,414,13.6,1.13
2,Karan,2026-02-14,29,1.0,0.08


## 4.2 Adding durations and offsets

In [15]:
d = pd.Timestamp("2026-03-15")

print(d + pd.Timedelta(days=10))                 # +10 days
print(d + pd.Timedelta(hours=36))                # +36 hours
print(d - pd.Timedelta(weeks=2))                 # -2 weeks

2026-03-25 00:00:00
2026-03-16 12:00:00
2026-03-01 00:00:00


`Timedelta` is an **exact** duration (days, hours). `DateOffset` is a **calendar** step (months, years) that follows the calendar rules:

In [16]:
jan31 = pd.Timestamp("2026-01-31")

print("+30 days   :", jan31 + pd.Timedelta(days=30))        # exactly 30 days later
print("+1 month   :", jan31 + pd.DateOffset(months=1))      # 28 Feb (no 31 Feb, so it stops at month end)
print("+1 year    :", pd.Timestamp("2024-02-29") + pd.DateOffset(years=1))      # leap day -> 28 Feb 2025

+30 days   : 2026-03-02 00:00:00
+1 month   : 2026-02-28 00:00:00
+1 year    : 2025-02-28 00:00:00


In [17]:
# Offsets on a whole column
people["next_review"] = people["joined"] + pd.DateOffset(years=1)
people[["name", "joined", "next_review"]]

,name,joined,next_review
0,Ravi,2024-06-10,2025-06-10
1,Asha,2025-01-25,2026-01-25
2,Karan,2026-02-14,2027-02-14


## 4.3 Business days

In [18]:
start = pd.Timestamp("2026-03-12")                       # a Thursday
print("start     :", start.day_name())
print("+3 b.days :", (start + pd.offsets.BDay(3)).strftime("%Y-%m-%d %A"))      # skips the weekend
print("business days between two dates:", np.busday_count("2026-03-01", "2026-03-31"))

start     : Thursday
+3 b.days : 2026-03-17 Tuesday
business days between two dates: 21


Holidays are not included automatically. For real business calendars pass a list of holidays (`np.busday_count(..., holidays=[...])`) or use the `holidays` library.

## 4.4 Comparing dates (filtering)

In [19]:
orders[orders["order_time"] >= "2026-03-08"]                           # text is understood as a date

,order_time,amount
4,2026-03-08 18:20:00,430
5,2026-03-13 11:00:00,310
6,2026-03-14 08:30:45,75
7,2026-03-15 20:10:00,540


In [20]:
orders[orders["order_time"].between("2026-03-05", "2026-03-08 23:59:59")]

,order_time,amount
2,2026-03-05 23:59:59,980
3,2026-03-06 00:05:10,60
4,2026-03-08 18:20:00,430


---
# 5. Selecting by date: `DatetimeIndex`

Setting the date column as the **index** unlocks powerful selection and the time-series tools (`resample`, `rolling("7D")`).

## Our main time series: daily demand for 2.5 years

Daily demand with a trend, a weekend effect, a yearly cycle, drops on a few public holidays and noise. (We use only fixed-date national holidays, 26 Jan, 15 Aug and 2 Oct. Festivals like Diwali or Holi change every year, so in real projects use an official list or the `holidays` library.)

In [21]:
rng = np.random.default_rng(13)
idx = pd.date_range("2024-01-01", "2026-06-30", freq="D")
t = np.arange(len(idx))

HOLIDAYS = pd.to_datetime([f"{y}-{md}" for y in (2024, 2025, 2026) for md in ("01-26", "08-15", "10-02")])

demand = (
    200
    + 0.06 * t                                            # slow upward trend
    + np.where(idx.dayofweek >= 5, 40, 0)                 # weekends are busier
    + 25 * np.sin(2 * np.pi * idx.dayofyear / 365.25)     # yearly seasonality
    + np.where(idx.isin(HOLIDAYS), -50, 0)                # holidays are quieter
    + rng.normal(0, 12, len(idx))                         # noise
).round(0)

daily = pd.Series(demand, index=idx, name="demand")
print(daily.index.dtype, "|", len(daily), "days")
daily.head()

datetime64[us] | 912 days


2024-01-01    222.0
2024-01-02    164.0
2024-01-03    213.0
2024-01-04    203.0
2024-01-05    218.0
Freq: D, Name: demand, dtype: float64

In [22]:
# Partial string indexing: pass a year, a month or a day as text
print(daily["2025"].shape)               # whole year
print(daily["2025-03"].shape)            # whole month
print(daily["2025-03-15"])               # one day

(365,)
(31,)
284.0


In [23]:
# Slices with dates are INCLUSIVE on both ends (like loc)
daily["2026-03-05":"2026-03-09"]

2026-03-05    277.0
2026-03-06    269.0
2026-03-07    304.0
2026-03-08    290.0
2026-03-09    265.0
Freq: D, Name: demand, dtype: float64

In [24]:
# Boolean conditions on the index
weekends = daily[daily.index.dayofweek >= 5]
print(len(weekends), "weekend days, average demand:", round(weekends.mean(), 1))
print("weekday average:", round(daily[daily.index.dayofweek < 5].mean(), 1))

260 weekend days, average demand: 271.0
weekday average: 230.2


## Time of day: `between_time`, `at_time`

For data with times inside the day (logs, sensor readings):

In [25]:
rng2 = np.random.default_rng(5)
events = pd.DataFrame({
    "timestamp": pd.Timestamp("2026-03-02") + pd.to_timedelta(np.sort(rng2.integers(0, 3 * 24 * 60, 400)), unit="min"),
    "amount": rng2.integers(50, 500, 400),
}).set_index("timestamp")

print(events.shape)
print("events 09:00-18:00:", len(events.between_time("09:00", "18:00")))
print("events exactly at 12:00:", len(events.at_time("12:00")))
events.head(3)

(400, 1)
events 09:00-18:00: 150
events exactly at 12:00: 0


,amount
timestamp,
2026-03-02 00:03:00,483
2026-03-02 00:04:00,91
2026-03-02 00:05:00,295


In [26]:
# The index must be sorted for slicing to work reliably
print(events.index.is_monotonic_increasing)
# If it is not: events = events.sort_index()

True


---
# 6. `resample`: change the frequency

`resample` is **groupby for time**: it groups the rows into time buckets (days, weeks, months) and aggregates.

- **Downsampling**: fine to coarse (daily -> monthly). Needs an aggregation (`sum`, `mean`, ...).
- **Upsampling**: coarse to fine (monthly -> daily). Needs a fill method.

In [27]:
# Monthly average and total demand
monthly = daily.resample("ME").agg(["mean", "sum", "max"]).round(0)
monthly.tail(4)

,mean,sum,max
2026-03-31,283.0,8769.0,330.0
2026-04-30,282.0,8475.0,330.0
2026-05-31,282.0,8747.0,339.0
2026-06-30,270.0,8110.0,329.0


In [28]:
# Weekly and quarterly
print(daily.resample("W").sum().tail(3))
print()
print(daily.resample("QE").mean().round(1).tail(4))

2026-06-21    1902.0
2026-06-28    1870.0
2026-07-05     518.0
Freq: W-SUN, Name: demand, dtype: float64

2025-09-30    232.4
2025-12-31    234.7
2026-03-31    274.7
2026-06-30    278.4
Freq: QE-DEC, Name: demand, dtype: float64


In [29]:
# Several aggregations with names
daily.resample("ME").agg(avg_demand="mean", peak="max", days="count").tail(3)

,avg_demand,peak,days
2026-04-30,282.500000,330.0,30
2026-05-31,282.161290,339.0,31
2026-06-30,270.333333,329.0,30


In [30]:
# Event data per hour: how many events and what total amount
per_hour = events["amount"].resample("h").agg(["count", "sum"])
per_hour.head(5)

,count,sum
timestamp,,
2026-03-02 00:00:00,7,2110
2026-03-02 01:00:00,7,1128
2026-03-02 02:00:00,8,2460
2026-03-02 03:00:00,8,2314
2026-03-02 04:00:00,7,2185


### Which side of the bucket? `closed` and `label`

`"W"` buckets end on **Sunday** and are **labelled by the end date**. Use `"W-MON"` for weeks that end on Monday, or `label="left"` to label by the start.

In [31]:
week = daily["2026-03-01":"2026-03-15"]
print(week.resample("W").sum())                     # 1 Mar is a Sunday: it forms a bucket alone
print()
print(week.resample("W", label="left", closed="left").sum())      # weeks start on Sunday

2026-03-01     309.0
2026-03-08    1950.0
2026-03-15    1965.0
Freq: W-SUN, Name: demand, dtype: float64

2026-03-01    1969.0
2026-03-08    1925.0
2026-03-15     330.0
Freq: W-SUN, Name: demand, dtype: float64


## Upsampling: monthly to daily

In [32]:
monthly_avg = daily.resample("ME").mean().round(0)

print(monthly_avg.head(3).resample("D").asfreq().head(3))          # asfreq: new rows are NaN
print()
print(monthly_avg.head(3).resample("D").ffill().iloc[[0, 1, 30, 31]])      # ffill: copy the last value forward

2024-01-31    220.0
2024-02-01      NaN
2024-02-02      NaN
Freq: D, Name: demand, dtype: float64

2024-01-31    220.0
2024-02-01    220.0
2024-03-01    233.0
2024-03-02    233.0
Name: demand, dtype: float64


## `resample` vs `groupby(Grouper)`

Both work. `resample` needs a datetime **index**. `groupby(pd.Grouper(key="date", freq="ME"))` works on a datetime **column** (chapter 10).

---
# 7. Missing dates and gaps

Real data skips days (no sales = no row). A time series with gaps breaks `shift`, `rolling` and models that assume **one row per day**.

In [33]:
gappy = daily["2026-05-01":"2026-05-10"].drop(pd.to_datetime(["2026-05-03", "2026-05-04", "2026-05-08"]))
print(len(gappy), "rows for 10 days")
gappy

7 rows for 10 days


2026-05-01    284.0
2026-05-02    301.0
2026-05-05    273.0
2026-05-06    265.0
2026-05-07    281.0
2026-05-09    315.0
2026-05-10    316.0
Name: demand, dtype: float64

In [34]:
# Rebuild the full calendar and see where the holes are
full_days = pd.date_range("2026-05-01", "2026-05-10", freq="D")
regular = gappy.reindex(full_days)
print("missing days:", regular[regular.isna()].index.strftime("%d %b").tolist())

missing days: ['03 May', '04 May', '08 May']


In [35]:
# Fill with what makes sense for the data:
print("no row = no sales (0)   :", regular.fillna(0).tolist())
print("carry last value forward:", regular.ffill().tolist())
print("straight line between  :", regular.interpolate().round(0).tolist())

no row = no sales (0)   : [284.0, 301.0, 0.0, 0.0, 273.0, 265.0, 281.0, 0.0, 315.0, 316.0]
carry last value forward: [284.0, 301.0, 301.0, 301.0, 273.0, 265.0, 281.0, 281.0, 315.0, 316.0]
straight line between  : [284.0, 301.0, 292.0, 282.0, 273.0, 265.0, 281.0, 298.0, 315.0, 316.0]


In [36]:
# Detect gaps in any series without knowing them in advance
gaps = gappy.index.to_series().diff()
gaps[gaps > pd.Timedelta(days=1)]

2026-05-05   3 days
2026-05-09   2 days
dtype: timedelta64[us]

Choose the fill by meaning: **0** when missing means "nothing happened" (sales, clicks), **ffill** for states (price, status), **interpolate** for smooth physical values (temperature).

# 8. Rolling windows by time, `shift`, `diff`

Chapter 09 used **row-count** windows (`rolling(7)`). On a **datetime index** you can use **time** windows (`rolling("7D")`), which stay correct even when rows are missing or irregular.

In [37]:
roll = pd.DataFrame({"demand": daily})
roll["avg_7d"] = daily.rolling("7D").mean().round(1)
roll["avg_28d"] = daily.rolling("28D").mean().round(1)
roll["same_day_last_week"] = daily.shift(7)                       # 7 rows back (valid because rows are daily and complete)
roll["change_vs_last_week_%"] = ((daily / daily.shift(7) - 1) * 100).round(1)
roll["2026-03-01":"2026-03-06"]

,demand,avg_7d,avg_28d,same_day_last_week,change_vs_last_week_%
2026-03-01,309.0,285.4,278.8,323.0,-4.3
2026-03-02,251.0,280.4,278.5,286.0,-12.2
2026-03-03,264.0,278.4,279.4,278.0,-5.0
2026-03-04,295.0,281.3,281.0,275.0,7.3
2026-03-05,277.0,280.6,281.1,282.0,-1.8
2026-03-06,269.0,280.6,280.6,269.0,0.0


In [38]:
# Irregular timestamps: events in the LAST hour, up to each event
events["amount"].rolling("1h").sum().head(5)

timestamp
2026-03-02 00:03:00     483.0
2026-03-02 00:04:00     574.0
2026-03-02 00:05:00     869.0
2026-03-02 00:11:00    1339.0
2026-03-02 00:24:00    1596.0
Name: amount, dtype: float64

In [39]:
# shift by TIME instead of rows: moves the index, not the values
print(daily.shift(7, freq="D").head(2))
print()
print(daily.diff(7).head(9).tail(2))              # value minus the value 7 rows earlier
print(daily.pct_change(7).round(3).iloc[7])

2024-01-08    222.0
2024-01-09    164.0
Freq: D, Name: demand, dtype: float64

2024-01-08   -18.0
2024-01-09    34.0
Freq: D, Name: demand, dtype: float64
-0.081


Row-based `shift(7)` means "7 rows back". It equals "7 days back" **only if** there is exactly one row per day. After you rebuilt the calendar (section 7), it does.

For ML features, remember (chapter 09): **shift before rolling**, so the window contains only the past.

---
# 9. Time zones

A timestamp can be **naive** (no zone, "10:00" but where?) or **aware** (10:00 in India, +05:30).

Best practice for data storage: keep everything in **UTC**, convert to **local time** only for display and for **local-time features** (like "evening rush hour").

In [40]:
# Server logs in UTC
utc_times = pd.Series(pd.to_datetime(["2026-06-01 18:30:00", "2026-06-01 23:45:00", "2026-06-02 03:15:00"]))

aware_utc = utc_times.dt.tz_localize("UTC")                  # say: these naive times ARE UTC
ist = aware_utc.dt.tz_convert("Asia/Kolkata")                # convert to Indian time

pd.DataFrame({"utc": aware_utc, "ist": ist, "hour_utc": aware_utc.dt.hour, "hour_ist": ist.dt.hour})

,utc,ist,hour_utc,hour_ist
0,2026-06-01 18:30:00+00:00,2026-06-02 00:00:00+05:30,18,0
1,2026-06-01 23:45:00+00:00,2026-06-02 05:15:00+05:30,23,5
2,2026-06-02 03:15:00+00:00,2026-06-02 08:45:00+05:30,3,8


- `tz_localize`: **attach** a zone to naive times (the clock time stays the same).
- `tz_convert`: **convert** aware times to another zone (the clock time changes, the moment stays the same).

Notice the day changed too: `2026-06-01 23:45 UTC` is `2026-06-02 05:15` in India. A "daily" aggregation on UTC and one on IST give **different results**.

In [41]:
# You cannot mix naive and aware timestamps
try:
    pd.Timestamp("2026-06-01 10:00") - aware_utc.iloc[0]
except TypeError as e:
    print("TypeError:", e)

TypeError: Cannot subtract tz-naive and tz-aware datetime-like objects.


`Asia/Kolkata` has no daylight saving time, but many zones do (`America/New_York`, `Europe/London`). On the days clocks change, a day has 23 or 25 hours, and some local times do not exist or happen twice. Another reason to **store in UTC**.

# 10. Date features for ML

Models cannot understand a raw date. We **extract features** that capture how time affects the target.

## 10.1 Calendar features

In [42]:
feat = pd.DataFrame({"demand": daily})
i = feat.index

feat["dow"] = i.dayofweek
feat["month"] = i.month
feat["day_of_year"] = i.dayofyear
feat["week_of_year"] = i.isocalendar().week.astype(int)
feat["is_weekend"] = (i.dayofweek >= 5).astype(int)
feat["is_month_start"] = i.is_month_start.astype(int)
feat["is_month_end"] = i.is_month_end.astype(int)
feat["quarter"] = i.quarter
feat["trend"] = np.arange(len(feat))                       # days since the start (captures long-term growth)
feat.head(3)

,demand,dow,month,day_of_year,week_of_year,is_weekend,is_month_start,is_month_end,quarter,trend
2024-01-01,222.0,0,1,1,1,0,1,0,1,0
2024-01-02,164.0,1,1,2,1,0,0,0,1,1
2024-01-03,213.0,2,1,3,1,0,0,0,1,2


## 10.2 Holiday flags and distance to events

In [65]:
feat["is_holiday"] = i.isin(HOLIDAYS).astype(int)
feat["day_before_holiday"] = i.isin(HOLIDAYS - pd.Timedelta(days=1)).astype(int)

# days until the next holiday
holiday_series = pd.Series(HOLIDAYS.sort_values(), name="h")
feat["days_to_next_holiday"] = [(holiday_series[holiday_series >= d].min() - d).days if (holiday_series >= d).any() else np.nan for d in i]

feat.loc["2025-08-12":"2025-08-17", ["demand", "is_holiday", "day_before_holiday", "days_to_next_holiday"]]

,demand,is_holiday,day_before_holiday,days_to_next_holiday
2025-08-12,214.0,0,0,3
2025-08-13,229.0,0,0,2
2025-08-14,221.0,0,1,1
2025-08-15,161.0,1,0,0
2025-08-16,264.0,0,0,47
2025-08-17,293.0,0,0,46


## 10.3 Cyclical encoding (sine and cosine)

Month `12` and month `1` are neighbours, but as numbers they are **11 apart**. Same for hour 23 and hour 0, or Sunday (6) and Monday (0). A model that treats the number as a straight line is confused.

Fix: put the cycle on a **circle** with two columns, `sin` and `cos`.

In [44]:
months = pd.DataFrame({"month": range(1, 13)})
months["sin"] = np.sin(2 * np.pi * months["month"] / 12)
months["cos"] = np.cos(2 * np.pi * months["month"] / 12)

def circle_distance(a, b):
    ra, rb = months.loc[a - 1], months.loc[b - 1]
    return round(float(np.hypot(ra["sin"] - rb["sin"], ra["cos"] - rb["cos"])), 2)

print("plain number distance  Dec(12) <-> Jan(1):", abs(12 - 1))
print("circle distance        Dec(12) <-> Jan(1):", circle_distance(12, 1))
print("circle distance        Jan(1)  <-> Jul(7):", circle_distance(1, 7), "(opposite sides of the year)")
months.round(2).head(4)

plain number distance  Dec(12) <-> Jan(1): 11
circle distance        Dec(12) <-> Jan(1): 0.52
circle distance        Jan(1)  <-> Jul(7): 2.0 (opposite sides of the year)


,month,sin,cos
0,1,0.50,0.87
1,2,0.87,0.50
2,3,1.00,0.00
3,4,0.87,-0.50


In [45]:
def add_cyclical(df, col, period):
    df[f"{col}_sin"] = np.sin(2 * np.pi * df[col] / period)
    df[f"{col}_cos"] = np.cos(2 * np.pi * df[col] / period)
    return df

feat = add_cyclical(feat, "dow", 7)
feat = add_cyclical(feat, "month", 12)
feat = add_cyclical(feat, "day_of_year", 365.25)

feat[["dow", "dow_sin", "dow_cos", "month", "month_sin", "month_cos"]].head(4).round(2)

,dow,dow_sin,dow_cos,month,month_sin,month_cos
2024-01-01,0,0.00,1.00,1,0.5,0.87
2024-01-02,1,0.78,0.62,1,0.5,0.87
2024-01-03,2,0.97,-0.22,1,0.5,0.87
2024-01-04,3,0.43,-0.90,1,0.5,0.87


Use cyclical encoding for **linear models and neural networks**. Tree models (random forest, gradient boosting) can split on plain numbers and often do fine without it, but it does not hurt.

For small cycles with different behaviour per value (day of week), **one-hot** (`get_dummies`) is another good option.

## 10.4 Lag and rolling features (past only)

Same rule as chapter 09: features must use **only information available before the day we predict**.

In [46]:
y = feat["demand"]

feat["lag_1"] = y.shift(1)
feat["lag_7"] = y.shift(7)
feat["lag_14"] = y.shift(14)
feat["roll7_past"] = y.shift(1).rolling(7).mean()          # shift FIRST, then roll
feat["roll28_past"] = y.shift(1).rolling(28).mean()
feat["roll7_std_past"] = y.shift(1).rolling(7).std()

feat["roll7_LEAKY"] = y.rolling(7).mean()                  # includes TODAY's value: leakage!
feat["roll3_centered_LEAKY"] = y.rolling(3, center=True).mean()      # centered window: yesterday + today + TOMORROW: leakage!

feat[["demand", "lag_1", "lag_7", "roll7_past", "roll7_LEAKY"]].iloc[28:32].round(1)

,demand,lag_1,lag_7,roll7_past,roll7_LEAKY
2024-01-29,196.0,268.0,216.0,226.4,223.6
2024-01-30,218.0,196.0,234.0,223.6,221.3
2024-01-31,227.0,218.0,230.0,221.3,220.9
2024-02-01,218.0,227.0,216.0,220.9,221.1


---
# 11. Mini AI/ML example: demand forecasting

## 11.1 The golden rule: split by time, never randomly

A random split puts days from **the future** into the training set and past days into the test set. The model "knows" what happens after the day it predicts. Real life never works like that.

```
 WRONG (random):  train: ## # ## # ## #     test: # #  #  (mixed through the whole timeline)
 RIGHT (time)  :  train: ###################  |  test: #######   (the past trains, the future tests)
```

We train on everything up to the end of 2025, and test on the first half of 2026.

In [47]:
data = feat.dropna(subset=["lag_1", "lag_7", "lag_14", "roll7_past", "roll28_past", "roll7_std_past"]).copy()      # drop the first rows without history

train = data[data.index < "2026-01-01"]
test = data[data.index >= "2026-01-01"]

print("train:", train.index.min().date(), "->", train.index.max().date(), "|", len(train), "days")
print("test :", test.index.min().date(), "->", test.index.max().date(), "|", len(test), "days")
print("no overlap:", train.index.max() < test.index.min())

train: 2024-01-29 -> 2025-12-31 | 703 days
test : 2026-01-01 -> 2026-06-30 | 181 days
no overlap: True


## 11.2 Baselines first

Before any model, measure **simple rules**. A model must beat them to be worth anything.

In [48]:
def evaluate(actual, predicted):
    err = actual - predicted
    return pd.Series({"MAE": err.abs().mean(), "MAPE_%": (err.abs() / actual).mean() * 100})

results = {}

results["naive: same as yesterday"] = evaluate(test["demand"], test["lag_1"])
results["seasonal naive: same day last week"] = evaluate(test["demand"], test["lag_7"])
results["average of the last 7 days"] = evaluate(test["demand"], test["roll7_past"])

pd.DataFrame(results).T.round(2)

,MAE,MAPE_%
naive: same as yesterday,20.26,7.38
seasonal naive: same day last week,14.51,5.32
average of the last 7 days,17.19,6.19


## 11.3 A simple model on calendar + lag features

A **linear regression** solved with NumPy (`lstsq`), so you see everything it uses. (In your ML folder you will use scikit-learn for this.)

In [49]:
def fit_predict(train, test, cols):
    A = np.column_stack([np.ones(len(train)), train[cols].to_numpy(float)])
    coef, *_ = np.linalg.lstsq(A, train["demand"].to_numpy(float), rcond=None)
    A_test = np.column_stack([np.ones(len(test)), test[cols].to_numpy(float)])
    return pd.Series(A_test @ coef, index=test.index), coef

dow_dummies = pd.get_dummies(data["dow"], prefix="dow", dtype=int).iloc[:, 1:]       # drop one to avoid duplicates
data2 = data.join(dow_dummies)
train2, test2 = data2[data2.index < "2026-01-01"], data2[data2.index >= "2026-01-01"]

past_features = ["trend", "lag_7", "roll28_past", "is_holiday", "month_sin", "month_cos"] + dow_dummies.columns.tolist()

pred, _ = fit_predict(train2, test2, past_features)
results["linear model (past-only features)"] = evaluate(test2["demand"], pred)

pd.DataFrame(results).T.round(2)

,MAE,MAPE_%
naive: same as yesterday,20.26,7.38
seasonal naive: same day last week,14.51,5.32
average of the last 7 days,17.19,6.19
linear model (past-only features),9.64,3.51


The model beats all baselines, because it knows about the weekend effect, holidays, the trend and the season.

## 11.4 What leakage looks like

Same model, one extra feature: `roll3_centered_LEAKY`. It is a 3-day average **centered** on the day (`rolling(3, center=True)`), a very common beginner mistake: the window contains yesterday, **today** and **tomorrow**. We compare both models on exactly the same rows.

In [50]:
leaky_cols = past_features + ["roll3_centered_LEAKY"]
tr_l = train2.dropna(subset=leaky_cols)
te_l = test2.dropna(subset=leaky_cols)

pred_honest, _ = fit_predict(tr_l, te_l, past_features)
pred_leaky, _ = fit_predict(tr_l, te_l, leaky_cols)

compare = pd.DataFrame({
    "honest model": evaluate(te_l["demand"], pred_honest),
    "model with leaky feature": evaluate(te_l["demand"], pred_leaky),
}).T.round(2)
compare

,MAE,MAPE_%
honest model,9.70,3.53
model with leaky feature,7.57,2.76


The leaky model looks clearly **better** (the error drops by about a fifth), but it is cheating: a third of that feature **is** today's demand and another third is tomorrow's. In production, tomorrow's demand is unknown, so this feature cannot exist and the real error will be the honest one or worse.

**How to catch leakage:** a suspiciously good score, a feature that is too important, and always asking *"will I know this value at prediction time?"*

## 11.5 Walk-forward validation (several time splits)

One split can be lucky. **Walk-forward** (expanding window) validation repeats: train on the past, test on the next block, move forward.

In [51]:
folds = [
    ("2025-01-01", "2025-04-01"),
    ("2025-04-01", "2025-07-01"),
    ("2025-07-01", "2025-10-01"),
    ("2025-10-01", "2026-01-01"),
]

rows = []
for start, end in folds:
    tr = data2[data2.index < start]                              # everything BEFORE the fold
    te = data2[(data2.index >= start) & (data2.index < end)]     # the next 3 months
    p, _ = fit_predict(tr, te, past_features)
    rows.append({
        "test period": f"{start} -> {end}",
        "train days": len(tr),
        "model MAE": (te["demand"] - p).abs().mean(),
        "seasonal naive MAE": (te["demand"] - te["lag_7"]).abs().mean(),
    })

pd.DataFrame(rows).round(2)

,test period,train days,model MAE,seasonal naive MAE
0,2025-01-01 -> 2025-04-01,338,10.67,15.01
1,2025-04-01 -> 2025-07-01,428,10.17,14.80
2,2025-07-01 -> 2025-10-01,519,8.60,14.07
3,2025-10-01 -> 2026-01-01,611,8.57,12.51


scikit-learn has this ready: `from sklearn.model_selection import TimeSeriesSplit`. The idea is identical.

**Time-series ML checklist:**
1. Sort by time, make the calendar regular (one row per period).
2. Features from the **past only** (shift before rolling).
3. **Split by time**. Walk-forward for a reliable estimate.
4. Compute **baselines** first.
5. Fit scalers, imputers and encoders on the **train** period only.
6. Aggregate on the **local** time zone that matches the business.

---
# 12. Pandas 3.x notes

- **Resolution:** pandas 3 chooses the time resolution from the input (`datetime64[us]` for normal strings, `[s]` when converting from seconds) instead of always nanoseconds. Dates outside the old 1677-2262 range now work. Do not assume `ns`.
- **Frequency aliases:** `"ME"`, `"QE"`, `"YE"`, `"h"`, `"min"`, `"s"`. The old `"M"`, `"Q"`, `"Y"`, `"H"`, `"T"` raise errors.
- **`dt.week` is removed**: use `dt.isocalendar().week`.
- `fillna(method="ffill")` is removed: use `.ffill()`.
- `to_datetime(..., format="mixed")` parses each value separately. With `dayfirst=True` it can misread ISO dates (chapter 08): parse ISO first.
- Naive and aware timestamps cannot be compared or subtracted (`TypeError`).
- `Series.dt` works on pandas datetime-like columns. On text you get an `AttributeError`.

---
# 13. Common mistakes

| Mistake | Fix |
|---|---|
| Date columns left as text | `pd.to_datetime` (or `parse_dates=` in `read_csv`) |
| Ambiguous `03/04/2026` parsed the wrong way | Give `format=` explicitly or `dayfirst=True` and verify |
| `dayfirst=True` with `format="mixed"` on ISO dates | Parse ISO first (chapter 08) |
| `resample("M")` / `freq="H"` errors | `"ME"`, `"h"` |
| `dt.week` | `dt.isocalendar().week` |
| Grouping by `dt.date` (object dtype) | `dt.normalize()` or `resample("D")` |
| Slicing an unsorted index | `sort_index()` first |
| Gaps in the calendar then `shift(7)` | Reindex to a full `date_range`, fill sensibly |
| Using row windows on irregular data | Time windows: `rolling("7D")` |
| Mixing naive and aware times | Localize everything, store in UTC |
| Daily totals computed in UTC for a local business | Convert to local time first |
| Forgetting that `Timedelta.days` ignores the hours | Use `total_seconds()` |
| Adding 30 days when you mean one month | `DateOffset(months=1)` |
| Hour/month/weekday as plain numbers for linear or NN models | Cyclical (sin/cos) or one-hot |
| **Random** train/test split on time data | Split by time, walk-forward validation |
| `rolling(7).mean()` as feature for today | `shift(1)` first |
| Fitting scalers/encoders on all data | Fit on the train period only |
| No baseline | Naive and seasonal-naive first |

---
# 14. Interview questions

**Q1. How do you convert a text column to dates and what if the format is mixed or wrong?**
`pd.to_datetime(col, format=..., errors="coerce")`. With a known format give `format=` (fast and safe). For mixed formats use `format="mixed"` (carefully, ISO first). Invalid values become `NaT`.

**Q2. What does `resample` do and how does it differ from `groupby`?**
`resample` groups rows into time buckets (day, week, month) on a datetime index and aggregates (downsampling), or fills new rows (upsampling). `groupby` can group by any key. `groupby(pd.Grouper(freq=...))` is the equivalent on a column.

**Q3. `Timedelta` vs `DateOffset`?**
`Timedelta` is an exact duration (hours, days). `DateOffset` follows calendar rules (add one month, business days).

**Q4. How do you handle missing dates in a time series?**
Reindex to the full `date_range` and fill by meaning: zero, forward fill, or interpolation. Then lags and rolling windows are correct.

**Q5. How would you extract features from a timestamp for ML?**
Hour, day of week, month, quarter, day of year, is_weekend, month start/end, holiday flags, days to/since an event, trend index, plus lag and rolling features. Cyclical features use sine and cosine.

**Q6. Why use sine/cosine encoding for hour or month?**
They are cyclic: hour 23 is next to hour 0, December next to January. Plain numbers hide that. Sin/cos puts them on a circle so neighbours stay close.

**Q7. Why must time-series data be split by time and not randomly?**
A random split trains on the future and tests on the past, so the model sees information it would not have in production and the score is overly optimistic. Use a chronological split or walk-forward validation.

**Q8. What is data leakage in time series and how do you avoid it?**
Using information from after the prediction time: rolling means that include today, global statistics computed on all data, future labels. Shift features before rolling, fit transformers on the train period only, and always ask if the value is available at prediction time.

**Q9. What is the difference between `tz_localize` and `tz_convert`?**
`tz_localize` attaches a time zone to naive times. `tz_convert` changes aware times from one zone to another (same instant, different clock).

**Q10. Why store timestamps in UTC?**
It is unambiguous, has no daylight saving jumps, and makes comparisons and joins across regions safe. Convert to local time only for display and local-time features.

**Q11. How do you compute a 7-day moving average when some days are missing?**
`series.rolling("7D").mean()` on a datetime index (time window), or reindex to a complete calendar first and then use `rolling(7)`.

**Q12. Why compute baselines in forecasting?**
They show the minimum that must be beaten. Naive (yesterday) and seasonal-naive (same weekday last week) are surprisingly strong. A complex model that cannot beat them is not useful.

---
# 15. Practice questions

Run this cell first to get the practice data.

In [52]:
rng = np.random.default_rng(21)
stamps = pd.Timestamp("2026-04-01") + pd.to_timedelta(np.sort(rng.integers(0, 14 * 24 * 60, 300)), unit="min")

log = pd.DataFrame({
    "timestamp": stamps.astype(str),
    "user": rng.choice(["u1", "u2", "u3", "u4"], 300),
    "amount": rng.integers(50, 500, 300),
})

sales_gap = pd.Series(
    [100, 110, 95, 120, 130, 90, 105],
    index=pd.to_datetime(["2026-05-01", "2026-05-02", "2026-05-05", "2026-05-06", "2026-05-07", "2026-05-09", "2026-05-10"]),
    name="sales",
)

utc_log = pd.Series(["2026-06-01 18:30:00", "2026-06-01 23:45:00", "2026-06-02 03:15:00"])

members = pd.DataFrame({
    "name": ["Ravi", "Asha", "Karan", "Neha"],
    "joined": ["2023-07-15", "2024-12-01", "2025-11-20", "2026-05-30"],
})
log.head()

,timestamp,user,amount
0,2026-04-01 00:05:00,u2,461
1,2026-04-01 02:11:00,u1,379
2,2026-04-01 02:58:00,u3,436
3,2026-04-01 03:28:00,u2,468
4,2026-04-01 03:59:00,u2,402


**Q1.** Convert `log["timestamp"]` to datetime. Add `hour` and `day_name` columns.
**Q2.** Count the events per hour of the day (0-23) and find the busiest hour.
**Q3.** Which day of the week has the highest total `amount`?
**Q4.** Select the events during business hours (09:00 to 18:00) on weekdays only.
**Q5.** Compute the time between consecutive events and its average in minutes.
**Q6.** `sales_gap` is missing some days. Rebuild the full daily calendar and fill the missing days with 0.
**Q7.** From the daily `amount` totals of the log, make a weekly total (weeks ending Sunday) and a table of totals per day.
**Q8.** On the filled `sales_gap` from Q6 compute a 3-day rolling mean and the difference from the previous day.
**Q9.** Convert `utc_log` to Indian time (`Asia/Kolkata`) and extract the local hour and local date.
**Q10.** For `members`, compute the tenure in days and in full months up to `2026-07-01`.
**Q11.** Add cyclical encoding (sin and cos) for the hour column of Q1.
**Q12.** Split the log by time: the **last 20% of days** (by calendar date) form the test set. Show that no day appears in both sets.

### Solutions

In [53]:
# Q1
lg = log.copy()
lg["timestamp"] = pd.to_datetime(lg["timestamp"])
lg["hour"] = lg["timestamp"].dt.hour
lg["day_name"] = lg["timestamp"].dt.day_name()
lg.head(3)

,timestamp,user,amount,hour,day_name
0,2026-04-01 00:05:00,u2,461,0,Wednesday
1,2026-04-01 02:11:00,u1,379,2,Wednesday
2,2026-04-01 02:58:00,u3,436,2,Wednesday


In [54]:
# Q2
per_hour = lg["hour"].value_counts().sort_index()
print("busiest hour:", per_hour.idxmax(), "with", per_hour.max(), "events")
per_hour.head(6)

busiest hour: 11 with 18 events


hour
0     7
1     8
2    15
3    11
4    15
5    14
Name: count, dtype: int64

In [55]:
# Q3
lg.groupby("day_name")["amount"].sum().sort_values(ascending=False)

day_name
Sunday       12794
Saturday     12622
Thursday     11896
Wednesday    11418
Friday       10711
Tuesday      10390
Monday       10056
Name: amount, dtype: int64

In [56]:
# Q4
indexed = lg.set_index("timestamp").sort_index()
biz = indexed.between_time("09:00", "18:00")
biz = biz[biz.index.dayofweek < 5]
print(len(biz), "of", len(indexed), "events")
biz.head(3)

86 of 300 events


,user,amount,hour,day_name
timestamp,,,,
2026-04-01 09:15:00,u2,329,9,Wednesday
2026-04-01 09:52:00,u4,139,9,Wednesday
2026-04-01 10:05:00,u2,427,10,Wednesday


In [57]:
# Q5
gaps = lg["timestamp"].sort_values().diff()
print("average gap (minutes):", round(gaps.dt.total_seconds().mean() / 60, 1))
gaps.describe()

average gap (minutes): 67.3


count                       299
mean     0 days 01:07:17.859531
std      0 days 01:12:43.509236
min             0 days 00:00:00
25%             0 days 00:23:30
50%             0 days 00:45:00
75%             0 days 01:31:00
max             0 days 11:37:00
Name: timestamp, dtype: object

In [58]:
# Q6
full = pd.date_range(sales_gap.index.min(), sales_gap.index.max(), freq="D")
filled = sales_gap.reindex(full, fill_value=0)
print(filled)

2026-05-01    100
2026-05-02    110
2026-05-03      0
2026-05-04      0
2026-05-05     95
2026-05-06    120
2026-05-07    130
2026-05-08      0
2026-05-09     90
2026-05-10    105
Freq: D, Name: sales, dtype: int64


In [59]:
# Q7
amount = indexed["amount"]
print(amount.resample("W").sum())
print()
amount.resample("D").sum().head(5)

timestamp
2026-04-05    28975
2026-04-12    40410
2026-04-19    10502
Freq: W-SUN, Name: amount, dtype: int64



timestamp
2026-04-01    7425
2026-04-02    4434
2026-04-03    4119
2026-04-04    6188
2026-04-05    6809
Freq: D, Name: amount, dtype: int64

In [60]:
# Q8 (use the filled series from Q6)
out = pd.DataFrame({"sales": filled})
out["roll3"] = filled.rolling(3).mean().round(1)
out["change_vs_prev_day"] = filled.diff()
out

,sales,roll3,change_vs_prev_day
2026-05-01,100,NaN,NaN
2026-05-02,110,NaN,10.0
2026-05-03,0,70.0,-110.0
2026-05-04,0,36.7,0.0
2026-05-05,95,31.7,95.0
2026-05-06,120,71.7,25.0
2026-05-07,130,115.0,10.0
2026-05-08,0,83.3,-130.0
2026-05-09,90,73.3,90.0
2026-05-10,105,65.0,15.0


In [61]:
# Q9
u = pd.to_datetime(utc_log).dt.tz_localize("UTC").dt.tz_convert("Asia/Kolkata")
pd.DataFrame({"ist": u, "local_hour": u.dt.hour, "local_date": u.dt.date})

,ist,local_hour,local_date
0,2026-06-02 00:00:00+05:30,0,2026-06-02
1,2026-06-02 05:15:00+05:30,5,2026-06-02
2,2026-06-02 08:45:00+05:30,8,2026-06-02


In [62]:
# Q10
m = members.copy()
m["joined"] = pd.to_datetime(m["joined"])
ref = pd.Timestamp("2026-07-01")

m["tenure_days"] = (ref - m["joined"]).dt.days
m["tenure_months"] = (ref.year - m["joined"].dt.year) * 12 + (ref.month - m["joined"].dt.month) - (ref.day < m["joined"].dt.day).astype(int)
m

,name,joined,tenure_days,tenure_months
0,Ravi,2023-07-15,1082,35
1,Asha,2024-12-01,577,19
2,Karan,2025-11-20,223,7
3,Neha,2026-05-30,32,1


In [63]:
# Q11
lg["hour_sin"] = np.sin(2 * np.pi * lg["hour"] / 24)
lg["hour_cos"] = np.cos(2 * np.pi * lg["hour"] / 24)
lg[lg["hour"].isin([23, 0, 12])][["hour", "hour_sin", "hour_cos"]].drop_duplicates().round(2).sort_values("hour")

,hour,hour_sin,hour_cos
0,0,0.00,1.00
17,12,0.00,-1.00
103,23,-0.26,0.97


In [64]:
# Q12
days = lg["timestamp"].dt.normalize()
unique_days = np.sort(days.unique())
cut = unique_days[int(len(unique_days) * 0.8)]               # first day of the test set

train_log = lg[days < cut]
test_log = lg[days >= cut]

print("train days:", days[days < cut].nunique(), "| test days:", days[days >= cut].nunique())
print("no day in both:", set(days[days < cut]).isdisjoint(set(days[days >= cut])))
print("train ends:", train_log["timestamp"].max(), "| test starts:", test_log["timestamp"].min())

train days: 11 | test days: 3
no day in both: True
train ends: 2026-04-11 23:38:00 | test starts: 2026-04-12 00:30:00


---
# 16. Cheat sheet

```python
# PARSE
pd.to_datetime(s)                                   pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
pd.to_datetime(s, unit="s")                         pd.read_csv(f, parse_dates=["date"])
pd.date_range("2026-01-01", periods=10, freq="D")   # D B W ME MS QE YE h min s

# PARTS (.dt)
s.dt.year  .month  .day  .hour  .dayofweek  .day_name()  .month_name()  .quarter  .dayofyear
s.dt.isocalendar().week     s.dt.is_month_end     s.dt.normalize()     s.dt.floor("h")
s.dt.strftime("%Y-%m-%d")                           # text, display only

# ARITHMETIC
(a - b).dt.days    (a - b).dt.total_seconds()
ts + pd.Timedelta(days=7)    ts + pd.DateOffset(months=1)    ts + pd.offsets.BDay(3)
np.busday_count("2026-03-01", "2026-03-31")

# SELECT (datetime index, sorted)
s["2026"]   s["2026-03"]   s["2026-03-05":"2026-03-09"]
df.between_time("09:00", "18:00")    df.at_time("12:00")
df[df["t"].between("2026-01-01", "2026-03-31")]

# RESAMPLE
s.resample("ME").agg(["sum", "mean"])      s.resample("W", label="left", closed="left").sum()
s.resample("D").asfreq()    s.resample("D").ffill()          # upsampling
df.groupby(pd.Grouper(key="date", freq="ME"))["x"].sum()

# GAPS
s.reindex(pd.date_range(s.index.min(), s.index.max(), freq="D"), fill_value=0)

# WINDOWS
s.rolling("7D").mean()    s.shift(7)    s.diff(7)    s.pct_change(7)    s.shift(1).rolling(7).mean()   # past only

# TIME ZONES
s.dt.tz_localize("UTC").dt.tz_convert("Asia/Kolkata")

# ML FEATURES
dow, month, is_weekend, is_holiday, trend, lags, past-only rolling
sin = np.sin(2 * np.pi * x / period);  cos = np.cos(2 * np.pi * x / period)
train = df[df.index < cut];  test = df[df.index >= cut]            # SPLIT BY TIME
# baselines first: naive (lag_1), seasonal naive (lag_7)
```

**Next chapter:** `14_performance_best_practices`: memory usage and dtypes (`category`, downcasting), vectorization, avoiding `apply`, chunking, `eval`/`query`, profiling with `%timeit`, method chaining style, and faster alternatives (Polars, DuckDB) when pandas is not enough.